> Os exemplos desta página usam dados sintéticos gerados pelo projeto. Não são observações ASTER nem permitem interpretar valores como reflectância, temperatura ou classes reais. Veja [os dados de exemplo](../../../../data/README.md).



# Capítulo 04 — Filtros, Kernels e Campos

> 📖 **Referência:** Canty (2019), Capítulo 04

Notas em `aulas/filtros_e_kernels.md` e `aulas/campos_aleatorios.md`.

---

## 1. Intuição

Uma imagem de satélite não é só uma tabela de números independentes: cada pixel “conversa” com os vizinhos. **Filtrar** é recombinar essa vizinhança — para suavizar ruído, preservar ou destacar bordas.

O **kernel** é a receita: pesos (média, Gaussiano) ou regra não linear (mediana). Em sensoriamento remoto isso aparece o tempo todo: pré-processamento antes de classificar, realce de limites de uso do solo, e — no caso SAR — a base conceitual para filtros de speckle adaptativos.

O capítulo também introduz a ideia de **campo aleatório**: a dependência espacial é estatística, não só geométrica. Kernels são a operação prática; campos aleatórios são o modelo que justifica por que a vizinhança importa.

## 2. Fundamentos matemáticos

### Convolução discreta

Para imagem $g$ e kernel $w$ de suporte $(2a+1)\times(2b+1)$:

$$
(g * w)(i,j) = \sum_{k=-a}^{a} \sum_{l=-b}^{b} w(k,l)\, g(i-k,\, j-l)
$$

### Exemplos de kernels

- **Média** $3\times 3$: $w_{kl} = 1/9$ (suaviza, borra bordas).
- **Gaussiano**: pesos $\propto \exp(-(k^2+l^2)/(2\sigma^2))$ (escala espacial via $\sigma$).
- **Sobel**: aproxima $\partial g/\partial x$ e $\partial g/\partial y$; magnitude $|G|=\sqrt{G_x^2+G_y^2}$.
- **Laplaciano**: aproxima $\nabla^2 g$ (segunda derivada; sensível a ruído).

A **mediana** não é convolução: ordena a janela e pega o valor central — boa contra outliers.

## 3. Implementação

Leitura do raster sintético (Cap. 1) e filtros via `scipy.ndimage`. Backend `Agg` para rodar headless.

In [ ]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "study_utils").is_dir())
EXEMPLO = REPO / "data" / "exemplos" / "exemplo_sintetico.tif"
OUT = REPO / "conteudo/analise-de-imagens/filtros/figuras"
OUT.mkdir(parents=True, exist_ok=True)
BAND = 1


def read_band(path, band=1):
    try:
        import rasterio

        with rasterio.open(path) as ds:
            return ds.read(band).astype(np.float64)
    except Exception:
        pass
    try:
        from osgeo import gdal
        from osgeo.gdalconst import GA_ReadOnly

        gdal.AllRegister()
        ds = gdal.Open(str(path), GA_ReadOnly)
        arr = ds.GetRasterBand(band).ReadAsArray().astype(np.float64)
        ds = None
        return arr
    except Exception:
        pass
    import tifffile

    data = tifffile.imread(str(path))
    if data.ndim == 2:
        return data.astype(np.float64)
    if data.shape[0] < data.shape[-1]:
        return data[band - 1].astype(np.float64)
    return data[:, :, band - 1].astype(np.float64)


def stretch(img, p=(2, 98)):
    lo, hi = np.percentile(img, p)
    if hi <= lo:
        return np.zeros_like(img)
    return np.clip((img - lo) / (hi - lo), 0, 1)


img = read_band(EXEMPLO, BAND)
print("shape:", img.shape, "min/max:", float(img.min()), float(img.max()))

In [ ]:
from scipy import ndimage

SIZE, SIGMA = 5, 1.5

mean = ndimage.uniform_filter(img, size=SIZE)
gauss = ndimage.gaussian_filter(img, sigma=SIGMA)
median = ndimage.median_filter(img, size=SIZE)

sx = ndimage.sobel(img, axis=1)
sy = ndimage.sobel(img, axis=0)
sobel_mag = np.hypot(sx, sy)
lap = np.abs(ndimage.laplace(img))

# Convolução explícita 3x3 (média) — mesma ideia da soma do livro
w_mean = np.ones((3, 3)) / 9.0
conv_manual = ndimage.convolve(img, w_mean, mode="reflect")
print("diferença média vs convolve 3x3:", float(np.mean(np.abs(mean - ndimage.uniform_filter(img, 3)))))

## 4. Visualizações

Saídas em `../figuras/`.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, title, data in zip(
    axes,
    ["Original", f"Média {SIZE}×{SIZE}", f"Gauss σ={SIGMA}", f"Mediana {SIZE}×{SIZE}"],
    [img, mean, gauss, median],
):
    ax.imshow(stretch(data), cmap="gray")
    ax.set_title(title)
    ax.axis("off")
fig.tight_layout()
fig.savefig(OUT / "nb04_filtros_suavizacao.png", dpi=150, bbox_inches="tight")
plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, title, data in zip(
    axes,
    ["Original", "Sobel |G|", "|Laplaciano|"],
    [stretch(img), stretch(sobel_mag), stretch(lap)],
):
    ax.imshow(data, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
fig.tight_layout()
fig.savefig(OUT / "nb04_bordas.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("PNGs salvos em", OUT.resolve())

## 5. Scripts do `legado/canty/auxil/` e nota de mestrado (SAR / speckle)

Filtros lineares clássicos assumem ruído aditivo. Em **SAR**, o speckle é multiplicativo: a variância cresce com a intensidade. O caminho natural no repositório:

```python
# Filtro Gamma MAP (intensidade polarizada) — requer ENL e ambiente do livro
# %run ../../legado/canty/auxil/scripts/gamma_filter.py [opções] infile enl
# %run ../../legado/canty/auxil/scripts/mmse_filter.py ...
```

Para o mestrado: use Cap. 4 para dominar vizinhança/kernel; aplique depois Gamma/MMSE nas cenas Sentinel-1 (VV/VH) já exploradas no Cap. 1. A ponte conceitual é a mesma — estatística local na janela — com modelo de ruído diferente.

## 6. Exercícios

```bash
python exercicios/ex04_1.py          # média / Gauss / mediana → figuras/filtros_suavizacao.png
python exercicios/ex04_2.py          # Sobel / Laplaciano → figuras/bordas_sobel_laplacian.png
```

Ambos aceitam caminho do GeoTIFF e `-b` para a banda; default aponta para o raster sintético do Cap. 1.